In [ ]:
import time
import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:5000


In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
print(f"Fetch done; MNIST: {mnist.data.shape[0]} samples, {mnist.data.shape[1]} features")
X, y = mnist.data, mnist.target.astype(int)

# I calculate metrics on test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

Fetch done; MNIST: 70000 samples, 784 features


In [3]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")

train=(52500, 784)  val=(10500, 784)  test=(7000, 784)


In [4]:
def train_and_log(hidden_layer_sizes=(100,), learning_rate_init=0.001, batch_size=128, solver="adam", max_epochs=25, run_name=None):
    # logs params and epoch wise metrics
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("solver", solver)
        mlflow.log_param("max_epochs", max_epochs)
        mlflow.log_param("n_train_samples", X_train.shape[0])

        model = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, learning_rate_init=learning_rate_init, batch_size=batch_size, solver=solver, max_iter=1, warm_start=True, random_state=42) # type: ignore

        start = time.time()
        for epoch in range(1, max_epochs + 1):
            model.fit(X_train, y_train)
            train_loss = model.loss_
            val_acc = accuracy_score(y_val, model.predict(X_val))

            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch) # type: ignore

        train_time = time.time() - start

        # test metrics
        test_preds = model.predict(X_test)
        test_acc = accuracy_score(y_test, test_preds)
        test_f1 = f1_score(y_test, test_preds, average="macro")

        mlflow.log_metric("test_accuracy", test_acc) # type: ignore
        mlflow.log_metric("test_f1_macro", test_f1) # type: ignore
        mlflow.log_metric("train_time_sec", train_time)
        mlflow.log_metric("final_train_loss", model.loss_)

        mlflow.set_tag("dataset", "mnist")
        mlflow.set_tag("model", "MLPClassifier")
        mlflow.sklearn.log_model( # type: ignore
        model, name="model", skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"])

        run_id = mlflow.active_run().info.run_id # type: ignore
        print(
            f"Logged run {run_id} [{run_name}] | "
            f"test_acc={test_acc:.4f}  test_f1={test_f1:.4f}  "
            f"final_train_loss={model.loss_:.4f}  time={train_time:.1f}s"
        )
        return run_id

In [5]:
learning_rates = [0.001, 0.01]
architectures = [(50,), (100,), (100, 50)]

sweep_run_ids = []
for lr in learning_rates:
    for arch in architectures:
        rid = train_and_log(
            hidden_layer_sizes=arch,
            learning_rate_init=lr,
            batch_size=128,
            max_epochs=25,
            run_name=f"mlp-lr{lr}-arch{'x'.join(map(str, arch))}"
        )
        print("Done with run_id:", rid)
        sweep_run_ids.append(rid)

print("\nSweep run IDs:", sweep_run_ids)

Logged run 8d3720e9d9ee4a8a83ae252ddcdacbd8 [mlp-lr0.001-arch50] | test_acc=0.9601  test_f1=0.9598  final_train_loss=0.0064  time=19.1s
🏃 View run mlp-lr0.001-arch50 at: http://localhost:5000/#/experiments/2/runs/8d3720e9d9ee4a8a83ae252ddcdacbd8
🧪 View experiment at: http://localhost:5000/#/experiments/2
Done with run_id: 8d3720e9d9ee4a8a83ae252ddcdacbd8
Logged run c7cbeec193ae499396856b3a4b6ef692 [mlp-lr0.001-arch100] | test_acc=0.9704  test_f1=0.9702  final_train_loss=0.0071  time=49.9s
🏃 View run mlp-lr0.001-arch100 at: http://localhost:5000/#/experiments/2/runs/c7cbeec193ae499396856b3a4b6ef692
🧪 View experiment at: http://localhost:5000/#/experiments/2
Done with run_id: c7cbeec193ae499396856b3a4b6ef692
Logged run a7376f5b57a54b68b9dc0850c3e2ead1 [mlp-lr0.001-arch100x50] | test_acc=0.9663  test_f1=0.9661  final_train_loss=0.0130  time=48.5s
🏃 View run mlp-lr0.001-arch100x50 at: http://localhost:5000/#/experiments/2/runs/a7376f5b57a54b68b9dc0850c3e2ead1
🧪 View experiment at: http://l

In [6]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp"],
    order_by=["metrics.test_accuracy DESC"],
)

display_cols = [
    c
    for c in runs_df.columns #type: ignore
    if c
    in (
        "run_id",
        "tags.mlflow.runName",
        "params.learning_rate_init",
        "params.hidden_layer_sizes",
        "metrics.test_accuracy",
        "metrics.test_f1_macro",
        "metrics.final_train_loss",
        "metrics.train_time_sec"
    )
]
print(runs_df[display_cols].head(10).to_string(index=False)) #type: ignore

best_run = runs_df.iloc[0] #type: ignore
print(
    f"\nBest run: {best_run['run_id']}  "
    f"(test_accuracy={best_run['metrics.test_accuracy']:.4f})"
)

                          run_id  metrics.test_f1_macro  metrics.train_time_sec  metrics.test_accuracy  metrics.final_train_loss params.hidden_layer_sizes params.learning_rate_init    tags.mlflow.runName
c7cbeec193ae499396856b3a4b6ef692               0.970172               49.863909               0.970429                  0.007141                    (100,)                     0.001    mlp-lr0.001-arch100
a7376f5b57a54b68b9dc0850c3e2ead1               0.966060               48.473439               0.966286                  0.013030                 (100, 50)                     0.001 mlp-lr0.001-arch100x50
8d3720e9d9ee4a8a83ae252ddcdacbd8               0.959832               19.060403               0.960143                  0.006446                     (50,)                     0.001     mlp-lr0.001-arch50
f2e6051d824c4e8ba887cb5eccfcf5fa               0.953998               24.833236               0.954286                  0.332158                     (50,)                      0.01    